# LING 539 Final Project
## By: Jakob Garcia

## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier

LABEL	DESCRIPTION   
0	Not a movie or TV show review  
1	A positive movie or TV show review  
2	A negative movie or TV show review

In [3]:
# loading the data and checking for class imbalance
data = pd.read_csv("csv/train.csv", encoding="utf-8")
data['LABEL'].value_counts()

LABEL
0    32289
1    19139
2    18889
Name: count, dtype: int64

In [4]:
data.head(5)

,ID,TEXT,LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0
1,11848336500074516230,Just getting released from a six month drug re...,1
2,8453485550425672763,I was greatly dissappointed when I popped this...,0
3,13088897910749354342,This is a film that has garnered any interest ...,2
4,4199320520317837800,This is one of the more adorable episodes of t...,1


Checking for NA values

In [5]:
data.isna().sum()

,0
ID,0
TEXT,7
LABEL,0


In [6]:
data[data['TEXT'].isna()]

,ID,TEXT,LABEL
525,8913186936642501596,NaN,0
10515,15945925172219367463,NaN,0
13306,15739941089251531985,NaN,0
23103,8542348540201902718,NaN,0
33325,10576592012817923472,NaN,0
41132,492629661749308733,NaN,0
55472,7811925937803840518,NaN,0


Filling all NaNs with text versions of it. Hopefully the model learns NaN means label 0 based on the data?

In [7]:
data['TEXT'] = data["TEXT"].fillna("NaN")

Creating a tf-idf vectorizer for unigram and bigrams

In [8]:
# filter out super rare words and stop words
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2)

## Multiclass Logistic Regression

In [9]:
x = vectorizer.fit_transform(data['TEXT'])
x_train, x_dev, y_train, y_dev = train_test_split(x, data['LABEL'], test_size=0.10, random_state=539)

I will fit a logistic regression model to predict the 3 classes. I performed hyperparameter tuning on the 'C' parameter using grid search

In [10]:
# param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
param_grid = {'C': [100, 200, 400, 1000]}

grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=539, class_weight="balanced", max_iter=500),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

# Search the grid for the best model
grid_search.fit(x_train, y_train)

GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced', max_iter=500,
                                          random_state=539),
             n_jobs=-1, param_grid={'C': [100, 200, 400, 1000]},
             scoring='f1_macro')

After the grid search is complete, we can inspect the best parameters found and the corresponding best F1 score.

In [11]:
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation f1_macro score: {grid_search.best_score_}")

# Get the best model from grid search
best_log_reg = grid_search.best_estimator_

# Evaluate the best model on the dev set
print(f"F1 score on development set with best model: {f1_score(best_log_reg.predict(x_dev), y_dev, average="macro")}")

Best parameters found: {'C': 100}
Best cross-validation f1_macro score: 0.921397326276192
F1 score on development set with best model: 0.9245120575913482


Checking the correctness of the model

In [12]:
confusion_matrix(y_dev, best_log_reg.predict(x_dev))

array([[3246,   31,   18],
       [  62, 1719,  151],
       [  36,  155, 1614]])

### Exporting this models predictions

In [4]:
test_data = pd.read_csv("csv/test.csv", encoding="utf-8")
test_data['TEXT'].isna().sum()

np.int64(0)

In [ ]:
test_x = vectorizer.transform(test_data['TEXT'])
test_y = best_log_reg.predict(test_x)
test_data['LABEL'] = test_y

In [ ]:
pd.Series(test_y).value_counts()

,count
0,8035
1,4895
2,4650


In [ ]:
csv = test_data.drop(columns='TEXT')
csv.to_csv("submission.csv", index=False)

## Two Binary Logistic Regressions

For this model, I split the classification task into two. Predict whether an input is a movie review or not, and then for the movie reviews, classify if it is positive or negative.

In [14]:
# Indicate if a row is a movie or not
data['MOVIE'] = (data['LABEL'] != 0).astype(int)

### Training the movie predictor

In [15]:
# Create a train test split for the model and the overall performance
x = vectorizer.fit_transform(data['TEXT'])
x_train, x_dev, y_train, y_dev = train_test_split(x, data['LABEL'], test_size=0.10, random_state=539)

x_train2, x_dev2, y_train2, y_dev2 = train_test_split(x_train, data.loc[y_train.index, 'MOVIE'], test_size=0.05, random_state=539)

# param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
param_grid = {'C': [100, 200, 400, 1000]}


grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=539, class_weight="balanced", solver="liblinear"),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid_search.fit(x_train2, y_train2)

GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          random_state=539,
                                          solver='liblinear'),
             n_jobs=-1, param_grid={'C': [100, 200, 400, 1000]},
             scoring='f1_macro')

In [16]:
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation f1_macro score: {grid_search.best_score_}")

# Get the best model
best_movie_log_reg = grid_search.best_estimator_

# Evaluate the best model on the dev set
print(f"F1 score on development set with best model: {f1_score(best_movie_log_reg.predict(x_dev2), y_dev2, average="macro")}")

Best parameters found: {'C': 1000}
Best cross-validation f1_macro score: 0.9793405413945269
F1 score on development set with best model: 0.9791221571192354


In [17]:
confusion_matrix(y_dev2, best_movie_log_reg.predict(x_dev2))

array([[1495,   13],
       [  53, 1604]])

### Training the positive/negative predictor

In [18]:
# filter to only actual movie reviews
movies = data.iloc[y_train.index]
movies = movies[movies['MOVIE'] == 1]
x_movie = vectorizer.transform(movies['TEXT'])

x_train3, x_dev3, y_train3, y_dev3 = train_test_split(x_movie, movies['LABEL'], test_size=0.05, random_state=539)

# param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
param_grid = {'C': [200, 300, 400]}

grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=539, class_weight="balanced", solver="liblinear"),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid_search.fit(x_train3, y_train3)

GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          random_state=539,
                                          solver='liblinear'),
             n_jobs=-1, param_grid={'C': [200, 300, 400]}, scoring='f1_macro')

In [19]:
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation f1_macro score: {grid_search.best_score_}")

# Get the best model
best_review_log_reg = grid_search.best_estimator_

# Evaluate the best model on the dev set
print(f"F1 score on development set with best model: {f1_score(best_review_log_reg.predict(x_dev3), y_dev3, average="macro")}")

Best parameters found: {'C': 400}
Best cross-validation f1_macro score: 0.9140411794052845
F1 score on development set with best model: 0.9136377617542741


In [20]:
confusion_matrix(y_dev3, best_review_log_reg.predict(x_dev3))

array([[807,  71],
       [ 77, 760]])

Overall performance

In [21]:
# Simulate new dataframe
test = data.copy().drop(columns=['LABEL', 'MOVIE']).iloc[y_dev.index]

In [22]:
# predict movie review or not
movies_pred = best_movie_log_reg.predict(x_dev)
test["LABEL"] = movies_pred
x2 = vectorizer.transform(test[test['LABEL'] == 1]['TEXT'])

In [23]:
# predict positive or negative
review_pred = best_review_log_reg.predict(x2)
test.loc[test['LABEL'] == 1, 'LABEL'] = review_pred

In [24]:
f1_score(test['LABEL'], y_dev, average="macro")
# 0.9288660515623245

0.9288660515623245

In [25]:
confusion_matrix(y_dev, test['LABEL'])

array([[3259,   24,   12],
       [  63, 1733,  136],
       [  39,  152, 1614]])

### Exporting this model's predictions

In [ ]:
test_data = pd.read_csv("csv/test.csv", encoding="utf-8")

In [ ]:
# predict if movie or not
test_x = vectorizer.transform(test_data['TEXT'])
test_movie = best_movie_log_reg.predict(test_x)
test_data['LABEL'] = test_movie

In [ ]:
# predict pos vs neg
test_x2 = vectorizer.transform(test_data[test_data['LABEL'] == 1]['TEXT'])
test_review = best_review_log_reg.predict(test_x2)
test_data.loc[test_data['LABEL'] == 1, 'LABEL'] = test_review

In [ ]:
test_data['LABEL'].value_counts()

,count
LABEL,
0,8074
1,4901
2,4605


In [ ]:
csv = test_data.drop(columns='TEXT')
csv.to_csv("submission2.csv", index=False)

## Ensembling Logistic Regressions



In [26]:
data['MOVIE'] = (data['LABEL'] != 0).astype(int)
# Create a train test split for the model and the overall performance
x = vectorizer.fit_transform(data['TEXT'])
x_train, x_dev, y_train, y_dev = train_test_split(x, data['LABEL'], test_size=0.10, random_state=539)

In [27]:
x_train2, x_dev2, y_train2, y_dev2 = train_test_split(x_train, data.loc[y_train.index, 'MOVIE'], test_size=0.05, random_state=539)

# Define multiple logistic regression models with different hyperparameters
clf1 = LogisticRegression(C=10, class_weight='balanced', random_state=539)
clf2 = LogisticRegression(C=100, class_weight='balanced', random_state=539)
clf3 = LogisticRegression(C=1000, class_weight='balanced', random_state=539)
clf4 = LogisticRegression(C=10000, class_weight='balanced', random_state=539)
clf5 = LogisticRegression(C=100000, class_weight='balanced', random_state=539)

# Create the ensemble
movie_voting_clf = VotingClassifier(
    estimators=[('lr1', clf1), ('lr2', clf2), ('lr3', clf3), ('lr4', clf4), ('lr5', clf5)],
    voting='soft'  # 'soft' uses probabilities, which is usually better for LR
)

# Fit the ensemble
movie_voting_clf.fit(x_train2, y_train2)

VotingClassifier(estimators=[('lr1',
                              LogisticRegression(C=10, class_weight='balanced',
                                                 random_state=539)),
                             ('lr2',
                              LogisticRegression(C=100, class_weight='balanced',
                                                 random_state=539)),
                             ('lr3',
                              LogisticRegression(C=1000,
                                                 class_weight='balanced',
                                                 random_state=539)),
                             ('lr4',
                              LogisticRegression(C=10000,
                                                 class_weight='balanced',
                                                 random_state=539)),
                             ('lr5',
                              LogisticRegression(C=100000,
                                                 class_weight='balanced',
                                                 random_state=539))],
                 voting='soft')

In [28]:
# Evaluate
ensemble_preds = movie_voting_clf.predict(x_dev2)
print(f"Ensemble F1 Macro Score: {f1_score(ensemble_preds, y_dev2, average='macro')}")

confusion_matrix(y_dev2, ensemble_preds)

Ensemble F1 Macro Score: 0.9772156454967205


array([[1484,   24],
       [  48, 1609]])

In [29]:
# filter to only actual movie reviews
movies = data.iloc[y_train.index]
movies = movies[movies['MOVIE'] == 1]
x_movie = vectorizer.transform(movies['TEXT'])

x_train3, x_dev3, y_train3, y_dev3 = train_test_split(x_movie, movies['LABEL'], test_size=0.05, random_state=539)

# Define multiple logistic regression models with different hyperparameters
clf1 = LogisticRegression(C=10, class_weight='balanced', random_state=539)
clf2 = LogisticRegression(C=100, class_weight='balanced', random_state=539)
clf3 = LogisticRegression(C=200, class_weight='balanced', random_state=539)
clf4 = LogisticRegression(C=1000, class_weight='balanced', random_state=539)
clf5 = LogisticRegression(C=100000, class_weight='balanced', random_state=539)

# clf1 = LogisticRegression(C=10, class_weight='balanced', random_state=539, solver="liblinear")
# clf2 = LogisticRegression(C=100, class_weight='balanced', random_state=539, solver="liblinear")
# clf3 = LogisticRegression(C=10, class_weight='balanced', random_state=539, solver="liblinear")

# Create the ensemble
review_voting_clf = VotingClassifier(
    estimators=[('lr1', clf1), ('lr2', clf2), ('lr3', clf3), ('lr4', clf4), ('lr5', clf5)],
    voting='soft'  # 'soft' uses probabilities, which is usually better for LR
)

# Fit the ensemble
review_voting_clf.fit(x_train3, y_train3)

VotingClassifier(estimators=[('lr1',
                              LogisticRegression(C=10, class_weight='balanced',
                                                 random_state=539)),
                             ('lr2',
                              LogisticRegression(C=100, class_weight='balanced',
                                                 random_state=539)),
                             ('lr3',
                              LogisticRegression(C=200, class_weight='balanced',
                                                 random_state=539)),
                             ('lr4',
                              LogisticRegression(C=1000,
                                                 class_weight='balanced',
                                                 random_state=539)),
                             ('lr5',
                              LogisticRegression(C=100000,
                                                 class_weight='balanced',
                                                 random_state=539))],
                 voting='soft')

In [30]:
# Evaluate
ensemble_preds = review_voting_clf.predict(x_dev3)
print(f"Ensemble F1 Macro Score: {f1_score(ensemble_preds, y_dev3, average='macro')}")

confusion_matrix(y_dev3, ensemble_preds)

Ensemble F1 Macro Score: 0.914820121760382


array([[805,  73],
       [ 73, 764]])

In [31]:
# Simulate new dataframe
test = data.copy().drop(columns=['LABEL', 'MOVIE']).iloc[y_dev.index]

# predict movie review or not
movies_pred = movie_voting_clf.predict(x_dev)
test["LABEL"] = movies_pred
x2 = vectorizer.transform(test[test['LABEL'] == 1]['TEXT'])

In [32]:
# predict positive or negative
review_pred = review_voting_clf.predict(x2)
test.loc[test['LABEL'] == 1, 'LABEL'] = review_pred

In [33]:
f1_score(test['LABEL'], y_dev, average="macro")

0.9320772647588728

In [34]:
confusion_matrix(y_dev, test['LABEL'])

array([[3249,   30,   16],
       [  51, 1744,  137],
       [  30,  143, 1632]])

Export

In [ ]:
test_data = pd.read_csv("csv/test.csv", encoding="utf-8")
# predict if movie or not
test_x = vectorizer.transform(test_data['TEXT'])
test_movie = movie_voting_clf.predict(test_x)
test_data['LABEL'] = test_movie

# predict pos vs neg
test_x2 = vectorizer.transform(test_data[test_data['LABEL'] == 1]['TEXT'])
test_review = review_voting_clf.predict(test_x2)
test_data.loc[test_data['LABEL'] == 1, 'LABEL'] = test_review

csv = test_data.drop(columns='TEXT')
csv.to_csv("submission3.csv", index=False)